In [ ]:
### This notebook contains the main codes used to generated captions and the embeddings. This notebook was large and I presented only the codes here which was written while working on the course project.

# `Boba LLM`

In [ ]:
from __future__ import annotations

from typing import Any, Dict, List, Optional, Sequence, Tuple
import numpy as np

def generate_non_acoustic_captions(
    acc_list: Optional[Sequence[Any]],
    ppg_list: Optional[Sequence[Any]],
    light_list: Optional[Sequence[Any]],
    gravity_list: Optional[Sequence[Any]],
    step_list: Optional[Sequence[Any]],
    *,
    window_seconds: float = 16.0,
    duty_cycle_seconds: float = 90.0,
    ppg_fs: float = 25.0,
    min_ppg_peaks: int = 3,
    step_aggregation: str = "max",
    decimals: int = 3,
) -> Dict[str, str]:
    """
    Generate concise SensorLM-style captions from one non-acoustic smartwatch sensing window.

    Expected inputs
    ---------------
    acc_list:
        Nx3 array-like with columns [X, Y, Z] from the Android accelerometer.
        Use None or an empty array/list when accelerometer data are unavailable.

    gravity_list:
        Nx3 array-like with columns [X, Y, Z] from the Android gravity sensor.
        Use None or an empty array/list when gravity is unavailable.

    ppg_list:
        1D array-like raw PPG values.
        Use None or [] when PPG is unavailable.

    light_list:
        1D array-like ambient light values.
        Use None or [] when light is unavailable.

    step_list:
        1D array-like values representing steps walked since the previous duty-cycle window.
        If empty or None, the caption treats this as likely no step-count change.
    """
    if window_seconds <= 0:
        raise ValueError("window_seconds must be positive.")
    if duty_cycle_seconds <= 0:
        raise ValueError("duty_cycle_seconds must be positive.")
    if ppg_fs <= 0:
        raise ValueError("ppg_fs must be positive.")
    if min_ppg_peaks < 1:
        raise ValueError("min_ppg_peaks must be at least 1.")

    # Missing data for any sensor should produce a missing-modality caption, not an exception.
    acc_xyz = _as_xyz_array(acc_list, sensor_name="accelerometer", allow_empty=True)
    gravity_xyz = _as_xyz_array(gravity_list, sensor_name="gravity", allow_empty=True)
    ppg = _as_finite_1d(ppg_list)
    light = _as_finite_1d(light_list)
    steps = _as_finite_1d(step_list)

    acc_caption = _caption_accelerometer(acc_xyz, decimals=decimals)
    gravity_caption = _caption_gravity(gravity_xyz, decimals=decimals)
    ppg_caption = _caption_ppg(
        ppg,
        sampling_rate=ppg_fs,
        min_ppg_peaks=min_ppg_peaks,
        window_seconds=window_seconds,
        decimals=decimals,
    )
    light_caption = _caption_light(light, decimals=decimals)
    step_caption = _caption_steps(
        steps,
        duty_cycle_seconds=duty_cycle_seconds,
        aggregation=step_aggregation,
        decimals=decimals,
    )

    merged_caption = (
        f"Non-acoustic smartwatch caption for one {window_seconds:g}-second sensing window. "
        "The step sensor summarizes walking since the previous duty-cycle window.\n\n"
        f"Accelerometer sensor caption:\n{acc_caption}\n\n"
        f"Gravity sensor caption:\n{gravity_caption}\n\n"
        f"PPG sensor caption:\n{ppg_caption}\n\n"
        f"Ambient light sensor caption:\n{light_caption}\n\n"
        f"Step sensor caption:\n{step_caption}\n\n"
        "Cross-sensor context: accelerometer captures device acceleration, gravity captures device orientation, "
        "PPG captures pulse-wave physiology, ambient light captures illumination context, and steps capture recent walking. "
        "These are non-acoustic contextual cues, not direct social-interaction labels."
    )

    return {
        "accelerometer_caption": acc_caption,
        "gravity_caption": gravity_caption,
        "ppg_caption": ppg_caption,
        "light_caption": light_caption,
        "step_caption": step_caption,
        "merged_caption": merged_caption,
    }

# ---------------------------------------------------------------------
# Sensor captions
# ---------------------------------------------------------------------

def _caption_accelerometer(acc_xyz: np.ndarray, *, decimals: int) -> str:
    if acc_xyz.shape[0] == 0:
        return (
            "Statistical: accelerometer features are unavailable. "
            "Structural: no acceleration pattern can be summarized. "
            "Semantic: wrist/device movement context is missing."
        )

    summary = _summarize_xyz(acc_xyz, include_axis_std=True)
    mag = summary["magnitude_features"]
    profile = summary["profile"]
    axis_std = summary["axis_std"]

    return (
        "Statistical: acceleration magnitude "
        f"mean={_fmt(mag.get('mean'), decimals)} m/s^2, "
        f"std={_fmt(mag.get('std'), decimals)} m/s^2, "
        f"range={_fmt(mag.get('range'), decimals)} m/s^2, "
        f"RMS={_fmt(mag.get('rms'), decimals)} m/s^2; "
        f"axis stds x/y/z={_fmt_triplet(axis_std, decimals)} m/s^2. "
        "Structural: "
        f"third-wise magnitude means={_fmt_list(profile['third_means'], decimals)}, "
        f"third-wise ranges={_fmt_list(profile['third_ranges'], decimals)}. "
        "Semantic: Android accelerometer values include gravity, so variation may reflect wrist motion, "
        "device repositioning, or orientation change; the gravity sensor gives the cleaner orientation cue."
    )


def _caption_gravity(gravity_xyz: np.ndarray, *, decimals: int) -> str:
    if gravity_xyz.shape[0] == 0:
        return (
            "Statistical: gravity features are unavailable. "
            "Structural: no gravity-vector orientation pattern can be summarized. "
            "Semantic: orientation context is missing for this window; treat this as missing gravity data, not as zero gravity."
        )

    summary = _summarize_xyz(gravity_xyz, include_axis_std=False)
    mag = summary["magnitude_features"]
    profile = summary["profile"]
    axis_mean = np.mean(gravity_xyz, axis=0)
    angle = _early_late_angle_degrees(gravity_xyz)

    return (
        "Statistical: gravity-vector magnitude "
        f"mean={_fmt(mag.get('mean'), decimals)} m/s^2, "
        f"std={_fmt(mag.get('std'), decimals)} m/s^2, "
        f"range={_fmt(mag.get('range'), decimals)} m/s^2; "
        f"mean vector x/y/z={_fmt_triplet(axis_mean, decimals)} m/s^2. "
        "Structural: "
        f"third-wise magnitude means={_fmt_list(profile['third_means'], decimals)}, "
        f"early-to-late gravity-vector angle={_fmt(angle, decimals)} degrees. "
        "Semantic: gravity describes device orientation relative to gravity; the angle summarizes wrist/device orientation change."
    )


def _caption_light(light: np.ndarray, *, decimals: int) -> str:
    if light.size == 0:
        return (
            "Statistical: ambient-light features are unavailable. "
            "Structural: no illumination pattern can be summarized. "
            "Semantic: local light context is missing."
        )

    feats = _basic_signal_features(light)
    profile = _thirds_profile(light)

    return (
        "Statistical: ambient light "
        f"mean={_fmt(feats.get('mean'), decimals)} lux, "
        f"std={_fmt(feats.get('std'), decimals)} lux, "
        f"range={_fmt(feats.get('range'), decimals)} lux, "
        f"IQR={_fmt(feats.get('iqr'), decimals)} lux. "
        "Structural: "
        f"third-wise light means={_fmt_list(profile['third_means'], decimals)}, "
        f"third-wise ranges={_fmt_list(profile['third_ranges'], decimals)}. "
        "Semantic: light summarizes local illumination; changes can reflect lighting transitions, sensor occlusion, "
        "or wrist movement relative to a light source."
    )


def _caption_steps(
    steps: np.ndarray,
    *,
    duty_cycle_seconds: float,
    aggregation: str,
    decimals: int,
) -> str:
    step_delta, source_text = _step_delta_from_values(steps, aggregation=aggregation)
    duty_cycle_minutes = duty_cycle_seconds / 60.0

    if steps.size == 0:
        return (
            "Statistical: no step-count update was observed; interpreted as likely 0 steps since the previous "
            f"{_fmt(duty_cycle_minutes, decimals)}-minute duty-cycle window. "
            "Structural: step count is a sparse duty-cycle summary, not a dense 16-second waveform. "
            "Semantic: likely no recent walking transition was captured by the step sensor."
        )

    return (
        f"Statistical: steps since the previous {_fmt(duty_cycle_minutes, decimals)}-minute duty-cycle window="
        f"{_fmt(step_delta, decimals)} ({source_text}). "
        "Structural: step count is a sparse mobility summary rather than a high-rate waveform. "
        "Semantic: nonzero steps indicate recent walking or ambulatory movement between sensing windows; "
        "this is important context for interpreting acceleration, PPG quality, and social context."
    )


def _caption_ppg(
    ppg: np.ndarray,
    *,
    sampling_rate: float,
    min_ppg_peaks: int,
    window_seconds: float,
    decimals: int,
) -> str:
    if ppg.size == 0:
        return _invalid_ppg_caption(
            reason="no raw PPG values were available",
        )

    try:
        import neurokit2 as nk
    except Exception as exc:
        raise ImportError(
            "NeuroKit2 is required for PPG processing. Install with: pip install neurokit2"
        ) from exc

    fs = int(round(sampling_rate))

    try:
        signals, info = nk.ppg_process(ppg, sampling_rate=fs)
    except Exception as exc:
        return _invalid_ppg_caption(
            reason=f"NeuroKit preprocessing failed ({type(exc).__name__})",
        )

    clean = _clean_column(signals, "PPG_Clean")
    rate = _clean_column(signals, "PPG_Rate")
    quality = _clean_column(signals, "PPG_Quality")
    peaks = _extract_ppg_peaks(signals, info, n_samples=ppg.size)

    if clean.size == 0:
        return _invalid_ppg_caption(
            reason="no finite cleaned PPG signal remained after preprocessing"
        )

    if peaks.size < min_ppg_peaks:
        return _invalid_ppg_caption(
            reason=f"only {peaks.size} PPG peak(s) were detected; at least {min_ppg_peaks} are required",
        )

    clean_feats = _array_summary(clean)
    rate_summary = _array_summary(rate)
    quality_summary = _array_summary(quality)
    peak_counts = _counts_by_thirds(peaks, ppg.size)

    hrv_results, _ = _extract_neurokit_hrv(peaks, sampling_rate=fs)
    hrv_text = _format_selected_hrv(hrv_results, decimals)

    rate_text = "pulse rate unavailable"
    if rate_summary:
        rate_text = (
            f"pulse rate mean={_fmt(rate_summary.get('mean'), decimals)} bpm, "
            f"std={_fmt(rate_summary.get('std'), decimals)} bpm"
        )

    quality_text = ""
    if quality_summary:
        quality_text = f"; PPG quality mean={_fmt(quality_summary.get('mean'), decimals)}"

    return (
        "Statistical: valid NeuroKit PPG; "
        f"detected peaks={peaks.size}; {rate_text}; "
        f"cleaned-signal range={_fmt(clean_feats.get('range'), decimals)}; "
        f"selected HRV={hrv_text}{quality_text}. "
        f"Structural: peak counts by early/middle/late window={peak_counts}. "
        "Semantic: PPG provides pulse-wave physiology and short-window beat-variability context; "
        f"HRV from a {window_seconds:g}-second window should be treated as descriptive context rather than a full autonomic assessment."
    )


def _invalid_ppg_caption(*, reason: str) -> str:
    return (
        f"Statistical: no valid PPG features after NeuroKit preprocessing; reason: {reason}. "
        "Structural: no reliable pulse-peak sequence was extracted. "
        "Semantic: treat PPG as missing or low quality; possible causes include motion artifact, poor skin contact, "
        "off-wrist placement, saturation, or missing data."
    )


# ---------------------------------------------------------------------
# Feature helpers
# ---------------------------------------------------------------------

def _summarize_xyz(xyz: np.ndarray, *, include_axis_std: bool) -> Dict[str, Any]:
    magnitude = np.linalg.norm(xyz, axis=1)

    out: Dict[str, Any] = {
        "magnitude": magnitude,
        "magnitude_features": _basic_signal_features(magnitude),
        "profile": _thirds_profile(magnitude),
    }

    if include_axis_std:
        out["axis_std"] = (
            np.std(xyz, axis=0, ddof=1)
            if xyz.shape[0] > 1
            else np.zeros(3, dtype=float)
        )

    return out


def _basic_signal_features(x: np.ndarray) -> Dict[str, float]:
    """
    Small, stable SKDH feature set: mean, std, range, IQR, RMS.

    This intentionally avoids a large feature bank because the generated caption should
    stay concise and embedding-friendly.
    """
    x = _as_finite_1d(x)

    if x.size == 0:
        return {}

    try:
        import skdh.features as skf
    except Exception as exc:
        raise ImportError(
            "scikit-digital-health is required for accelerometer, gravity, and light features. "
            "Install with: pip install scikit-digital-health"
        ) from exc

    feature_objects = {
        "mean": skf.Mean(),
        "std": skf.StdDev(),
        "range": skf.Range(),
        "iqr": skf.IQR(),
        "rms": skf.RMS(),
    }

    feats: Dict[str, float] = {}

    for name, obj in feature_objects.items():
        try:
            value = _first_finite_scalar(obj.compute(x))

            if np.isfinite(value):
                feats[name] = value

        except Exception:
            # Skip a failed feature rather than failing the whole caption.
            continue

    return feats


def _thirds_profile(x: np.ndarray) -> Dict[str, List[float]]:
    x = _as_finite_1d(x)

    if x.size == 0:
        return {
            "third_means": [],
            "third_ranges": [],
        }

    parts = [
        part
        for part in np.array_split(x, 3)
        if part.size > 0
    ]

    return {
        "third_means": [
            float(np.mean(part))
            for part in parts
        ],
        "third_ranges": [
            float(np.max(part) - np.min(part))
            for part in parts
        ],
    }


def _early_late_angle_degrees(xyz: np.ndarray) -> float:
    xyz = np.asarray(xyz, dtype=float)

    if xyz.ndim != 2 or xyz.shape[1] != 3 or xyz.shape[0] < 2:
        return float("nan")

    parts = [
        part
        for part in np.array_split(xyz, 3)
        if part.shape[0] > 0
    ]

    early = np.mean(parts[0], axis=0)
    late = np.mean(parts[-1], axis=0)

    denom = float(np.linalg.norm(early) * np.linalg.norm(late))

    if denom <= 0 or not np.isfinite(denom):
        return float("nan")

    cosine = float(np.dot(early, late) / denom)
    cosine = float(np.clip(cosine, -1.0, 1.0))

    return float(np.degrees(np.arccos(cosine)))


def _extract_neurokit_hrv(
    peaks: np.ndarray,
    *,
    sampling_rate: int,
) -> Tuple[Dict[str, Dict[str, float]], Dict[str, str]]:
    import neurokit2 as nk

    calls = {
        "time": lambda: nk.hrv_time(
            peaks,
            sampling_rate=sampling_rate,
            show=False,
        ),
        "frequency": lambda: nk.hrv_frequency(
            peaks,
            sampling_rate=sampling_rate,
            show=False,
            silent=True,
        ),
        "nonlinear": lambda: nk.hrv_nonlinear(
            peaks,
            sampling_rate=sampling_rate,
            show=False,
        ),
    }

    results: Dict[str, Dict[str, float]] = {}
    errors: Dict[str, str] = {}

    for name, fn in calls.items():
        try:
            features = _flatten_table(fn())
            finite = {
                key: value
                for key, value in features.items()
                if np.isfinite(value)
            }

            if finite:
                results[name] = finite

        except Exception as exc:
            errors[name] = f"{type(exc).__name__}: {exc}"

    return results, errors


def _format_selected_hrv(
    hrv: Dict[str, Dict[str, float]],
    decimals: int,
) -> str:
    selected = [
        ("time", "HRV_MeanNN"),
        ("time", "HRV_SDNN"),
        ("time", "HRV_RMSSD"),
        ("time", "HRV_pNN20"),
        ("frequency", "HRV_LF"),
        ("frequency", "HRV_HF"),
        ("frequency", "HRV_LFHF"),
        ("nonlinear", "HRV_SD1"),
        ("nonlinear", "HRV_SD2"),
        ("nonlinear", "HRV_SampEn"),
    ]

    parts: List[str] = []

    for domain, key in selected:
        value = hrv.get(domain, {}).get(key, np.nan)

        if np.isfinite(value):
            parts.append(f"{key}={_fmt(value, decimals)}")

    return ", ".join(parts) if parts else "none available"


# ---------------------------------------------------------------------
# Parsing / cleaning helpers
# ---------------------------------------------------------------------
def _as_xyz_array(
    records: Optional[Sequence[Any]],
    *,
    sensor_name: str,
    allow_empty: bool = False,
) -> np.ndarray:
    """
    Convert an [X, Y, Z] sensor input into a finite NumPy array.

    If allow_empty=True:
        None, [], np.empty((0, 3)), or all-NaN rows return np.empty((0, 3)).
        This lets the caption function generate a missing-modality caption.

    If allow_empty=False:
        empty input still raises an error.

    Invalid non-empty shapes still raise errors because they indicate a real data-format bug.
    """
    if records is None:
        if allow_empty:
            return np.empty((0, 3), dtype=float)
        raise ValueError(f"{sensor_name} data cannot be None.")

    arr = np.asarray(records, dtype=float)

    if arr.size == 0:
        if allow_empty:
            return np.empty((0, 3), dtype=float)
        raise ValueError(f"{sensor_name} data are empty.")

    if arr.ndim != 2 or arr.shape[1] != 3:
        raise ValueError(
            f"{sensor_name} data must be a numeric array with shape (n_samples, 3), columns [X, Y, Z]. "
            f"Received shape {arr.shape}."
        )

    arr = arr[np.all(np.isfinite(arr), axis=1)]

    if arr.shape[0] == 0:
        if allow_empty:
            return np.empty((0, 3), dtype=float)
        raise ValueError(f"{sensor_name} data contain no finite [X, Y, Z] rows.")

    return arr


def _as_finite_1d(values: Optional[Sequence[Any]]) -> np.ndarray:
    if values is None:
        return np.array([], dtype=float)

    arr = np.asarray(values, dtype=float).reshape(-1)

    return arr[np.isfinite(arr)]


def _step_delta_from_values(
    values: np.ndarray,
    *,
    aggregation: str,
) -> Tuple[float, str]:
    values = _as_finite_1d(values)

    if values.size == 0:
        return 0.0, "missing interpreted as likely no step change"

    if aggregation == "max":
        return float(np.max(values)), "maximum observed value used"

    if aggregation == "last":
        return float(values[-1]), "last observed value used"

    if aggregation == "sum":
        return float(np.sum(values)), "sum of observed values used"

    raise ValueError("step_aggregation must be one of: 'max', 'last', or 'sum'.")


def _clean_column(table: Any, column: str) -> np.ndarray:
    if not hasattr(table, "columns") or column not in table.columns:
        return np.array([], dtype=float)

    return _as_finite_1d(table[column].to_numpy())


def _extract_ppg_peaks(
    signals: Any,
    info: Dict[str, Any],
    *,
    n_samples: int,
) -> np.ndarray:
    if isinstance(info, dict) and "PPG_Peaks" in info:
        peaks = np.asarray(info["PPG_Peaks"], dtype=int).reshape(-1)

    elif hasattr(signals, "columns") and "PPG_Peaks" in signals.columns:
        peaks = np.flatnonzero(
            np.asarray(signals["PPG_Peaks"], dtype=float) == 1
        )

    else:
        peaks = np.array([], dtype=int)

    peaks = peaks[
        (peaks >= 0)
        & (peaks < n_samples)
    ]

    return np.unique(peaks)


def _counts_by_thirds(
    indices: np.ndarray,
    n_samples: int,
) -> List[int]:
    if n_samples <= 0:
        return [0, 0, 0]

    edges = np.linspace(0, n_samples, 4)
    counts: List[int] = []

    for i in range(3):
        if i == 2:
            mask = (
                (indices >= edges[i])
                & (indices <= edges[i + 1])
            )
        else:
            mask = (
                (indices >= edges[i])
                & (indices < edges[i + 1])
            )

        counts.append(int(np.sum(mask)))

    return counts


def _array_summary(x: np.ndarray) -> Dict[str, float]:
    x = _as_finite_1d(x)

    if x.size == 0:
        return {}

    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "range": float(np.max(x) - np.min(x)),
    }


def _flatten_table(table: Any) -> Dict[str, float]:
    if table is None:
        return {}

    if hasattr(table, "iloc"):
        if len(table) == 0:
            return {}

        row = table.iloc[0].to_dict()

    elif isinstance(table, dict):
        row = table

    else:
        return {}

    out: Dict[str, float] = {}

    for key, value in row.items():
        scalar = _first_finite_scalar(value)

        if np.isfinite(scalar):
            out[str(key)] = scalar

    return out


def _first_finite_scalar(value: Any) -> float:
    try:
        arr = np.asarray(value, dtype=float).reshape(-1)

    except Exception:
        return float("nan")

    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return float("nan")

    return float(arr[0])


# ---------------------------------------------------------------------
# Formatting helpers
# ---------------------------------------------------------------------

def _fmt(value: Any, decimals: int) -> str:
    try:
        v = float(value)

    except Exception:
        return "NA"

    if not np.isfinite(v):
        return "NA"

    return f"{v:.{decimals}f}"


def _fmt_list(
    values: Sequence[Any],
    decimals: int,
) -> str:
    return "[" + ", ".join(_fmt(v, decimals) for v in values) + "]"


def _fmt_triplet(
    values: Sequence[Any],
    decimals: int,
) -> str:
    arr = np.asarray(values, dtype=float).reshape(-1)

    if arr.size != 3:
        return "[NA, NA, NA]"

    return "[" + ", ".join(_fmt(v, decimals) for v in arr) + "]"

## Main

In [ ]:
# Checked: 1
# Resampling per probe aligns with our goal where the input will be per probe to find whether there is an interaction or not.
def resample_to_hz_per_probe(df, time_col, value_col_list, target_hz, agg="mean", interpolate=True, n_samples=-1): # e.g., n_samples: n expected samples
    period = pd.to_timedelta(1000.0 / target_hz, unit="ms")
    df_res = df[[time_col]+ value_col_list].copy()
    df_res = df_res.sort_values(by=time_col).reset_index(drop=True) # Resetting so important here, as there are index based work in this function
    df_res.set_index(time_col, inplace = True)
    df_res = df_res.resample(period, origin = 'start').agg(agg) # origin='start': origin is the first value of the timeseries https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html

    new_index = pd.date_range(start=df_res.index[0], periods=n_samples, freq=period) # [Fine, as I reset index before] Try not to use index. Index is bad to me :(
    df_res = df_res.reindex(new_index)

    if interpolate:
        # if df_res[value_col_list].isna().sum().sum() > 0: print(f'There are {df_res[value_col_list].isna().sum().sum()} rows containing missing values in the {value_col_list} column. I have interpolated to have them.')
        df_res = df_res.interpolate(method="time") # as it considers the difference between timestamps of samples. In real-world sensing data, there can have a variation always. But I think as we are resampling where the time difference between two data points will be equal, it will be similar to linear interpolation.
    return df_res.reset_index(drop=True) # if you want the T column, no need to drop the index.



# Checked: 1
def get_windowed_data(start_dt, df_sensor, sensor_name, bool_resample):
    window_end_dt = start_dt + pd.Timedelta(seconds=Data_collection_window_in_Sec)
    sub_df_sensor = df_sensor[(df_sensor['T'] >= start_dt) & (df_sensor['T'] <= window_end_dt)].copy()

    if (sensor_name == str_file_accel) or (sensor_name == str_file_gravity):
        value_col_list = ['X', 'Y', 'Z']
    else:
        value_col_list = dic_col_name.get(sensor_name)

    if bool_resample and (sensor_name != str_file_log_mel_spec) and (sub_df_sensor.shape[0] > 0): # [Fixed]: why should not the threshold of > 0 be aligned with minimum # of samples of image generation? Answer on dec 21: I think it depends on the application. If spectrogram needs n samples, it will set that condition. We will just return the data of a window and then, the function calling this function will use the data accordingly. We do not care how many samples they will need.
        n_samples = dict_sensor_sampling_rate.get(sensor_name) * Data_collection_window_in_Sec
        sub_df_sensor = resample_to_hz_per_probe(df= sub_df_sensor, time_col='T', value_col_list= value_col_list,
                                                target_hz= dict_sensor_sampling_rate.get(sensor_name), agg='mean', interpolate=True,
                                                n_samples = n_samples)
    return get_processed_data(sub_df_sensor, sensor_name)


# Checked: 1
def get_processed_data(sub_df_sensor_data, name_sensor_file):
    """Remember (for me): 🐝🐝🐝 This function is changed from the version of SocialPulse."""

    global crt_nperseg
    list_sensor_data = []
    
    if (name_sensor_file == str_file_accel) or (name_sensor_file == str_file_gravity):
        list_sensor_data =  sub_df_sensor_data[["X", "Y", "Z"]].astype(float).to_numpy()
    elif name_sensor_file == str_file_ppg:
        list_sensor_data = sub_df_sensor_data[ dic_col_name.get(str_file_ppg)[0] ].tolist()
    elif name_sensor_file == str_file_light:
        list_sensor_data = sub_df_sensor_data[ dic_col_name.get(str_file_light)[0] ].tolist()
    elif name_sensor_file == str_file_sTep_NOT_sleep:
        list_sensor_data = sub_df_sensor_data[ dic_col_name.get(str_file_sTep_NOT_sleep)[0] ].tolist()

    return list_sensor_data




# TODO: how will you handle this? Sometimes, probe starting times were not recorded as I found by exploring. Here, I am including the probe starting times for which we have audio data. The reason for prioratizing audio is due to it's performance.
def fix_probe_start_if_there_is_multiple(pid, df_sensor_start, bool_print_msg = True): # 1 P: 1 participant :)
    
    df_sensor_start[str_round_T_min] = df_sensor_start['T'].dt.floor('min')
    df_sensor_start = remove_rows_if_two_probes_started_super_close(pid, df_sensor_start, column_dt='T')

    return df_sensor_start[['T']]






def compute_steps_walked(
    df: pd.DataFrame,
    time_col: str,
    step_col: str,
    output_col: str = "steps_walked"
):
    """
    Compute steps walked from cumulative step-counter data.

    Logic:
    - current > previous: steps walked = current - previous
    - current == previous: steps walked = 0
    - current < previous: steps walked = current

    The first row is dropped because we do not know the change from the previous value.
    """

    if df is None or len(df) == 0:
        return pd.DataFrame(columns=[time_col, step_col])

    temp = df[[time_col, step_col]].copy()
    temp = temp.sort_values(time_col).reset_index(drop=True)

    if len(temp) <= 1:
        return pd.DataFrame(columns=[time_col, step_col])

    # Previous cumulative step count
    temp["previous_steps"] = temp[step_col].shift(1)

    # Raw difference between current and previous
    temp["step_diff"] = temp[step_col] - temp["previous_steps"]

    # Initialize output
    temp[output_col] = 0

    # Case 1: current > previous
    mask_increase = temp["step_diff"] > 0
    temp.loc[mask_increase, output_col] = temp.loc[mask_increase, "step_diff"]

    # Case 2: current == previous
    mask_same = temp["step_diff"] == 0
    temp.loc[mask_same, output_col] = 0

    # Case 3: current < previous
    mask_reset = temp["step_diff"] < 0
    temp.loc[mask_reset, output_col] = temp.loc[mask_reset, step_col]

    # Now drop the first row because its change is unknown
    temp = temp.iloc[1:].copy()

    # Replace cumulative step count with walked steps
    temp[step_col] = temp[output_col].copy()

    return temp[[time_col, step_col]].reset_index(drop=True)



def boba():
    list_caption_dics = []

    for pid in sorted(os.listdir(root_target_folder)):

        if 'DS_' in pid: continue

        df_start = get_pkl_file_sorted(pid, str_file_sensor_start)
        df_start = fix_probe_start_if_there_is_multiple(pid, df_start, bool_print_msg=False)

        df_acc = get_pkl_file_sorted(pid, str_file_accel)
        df_grav = get_pkl_file_sorted(pid, str_file_gravity)
        df_ppg = get_pkl_file_sorted(pid, str_file_ppg)
        df_light = get_pkl_file_sorted(pid, str_file_light)

        df_step =  get_pkl_file_sorted(pid, str_file_sTep_NOT_sleep)
        df_step =  compute_steps_walked( df_step,
                                        time_col='T',
                                        step_col= dic_col_name.get(str_file_sTep_NOT_sleep)[0])
        # print('\n\n\n ', pid)
        # display(HTML(df_step.to_html()))

        # # 🐝🐝 TODO: remove all probe start time which started in less than < 15 seconds (or maybe in < 1 minute)

        for __, row_start in df_start.iterrows():
            
            ts = row_start['T']

            list_acc = get_windowed_data(ts, df_acc, 
                                         sensor_name = str_file_accel,
                                         bool_resample = True)
            list_grav = get_windowed_data(ts, df_grav, 
                                         sensor_name = str_file_gravity,
                                         bool_resample = True)
            list_ppg = get_windowed_data(ts, df_ppg, 
                                         sensor_name = str_file_ppg,
                                         bool_resample = True)
            list_light = get_windowed_data(ts, df_light, 
                                         sensor_name = str_file_light,
                                         bool_resample = True)
            list_step = get_windowed_data(ts, df_step,
                                          sensor_name= str_file_sTep_NOT_sleep,
                                          bool_resample=False) # BOOL resample: False
            
            dict_caption = generate_non_acoustic_captions(acc_list= list_acc,
                                                          gravity_list= list_grav,
                                                          ppg_list= list_ppg,
                                                          light_list= list_light,
                                                          step_list= list_step,
                                                          step_aggregation="sum")
            
            dict_caption[str_parti_id] = pid
            dict_caption['T'] = ts
            list_caption_dics.append(dict_caption)
    
        final_df = pd.DataFrame(list_caption_dics)
        final_df.to_pickle(path_j(root_loc_findings, 'captions.pkl'))

boba()

In [ ]:
import pandas as pd
import os
pd.read_pickle("... /Katha/Social Interaction/Boba/captions.pkl")['merged_caption'].iloc[0]

'Non-acoustic smartwatch caption for one 16-second sensing window. The step sensor summarizes walking since the previous duty-cycle window.\n\nAccelerometer sensor caption:\nStatistical: acceleration magnitude mean=9.791 m/s^2, std=0.422 m/s^2, range=4.686 m/s^2, RMS=0.422 m/s^2; axis stds x/y/z=[0.318, 0.717, 0.658] m/s^2. Structural: third-wise magnitude means=[9.689, 9.829, 9.854], third-wise ranges=[4.056, 0.306, 1.222]. Semantic: Android accelerometer values include gravity, so variation may reflect wrist motion, device repositioning, or orientation change; the gravity sensor gives the cleaner orientation cue.\n\nGravity sensor caption:\nStatistical: gravity-vector magnitude mean=9.807 m/s^2, std=0.000 m/s^2, range=0.003 m/s^2; mean vector x/y/z=[-3.626, 1.867, 8.839] m/s^2. Structural: third-wise magnitude means=[9.807, 9.807, 9.807], early-to-late gravity-vector angle=15.495 degrees. Semantic: gravity describes device orientation relative to gravity; the angle summarizes wrist/d